In [ ]:
# PATH FIX — ensures all modules are found regardless of run location
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

In [ ]:
# IMPORTS

from data.ticker_list import tickers
from data.prices import dwn_adj_close_price
from src.returns import cal_logs
from src.his_returns import get_hist_returns
from src.x_returns import xday_return_calc
from src.var_historical import cal_var_s
from src.backtest import rolling_var_backtest
import pandas as pd
import numpy as np
import datetime as dt

In [ ]:
# USER INPUTS

print("Please enter these values...........")
years = int(input("Enter Years: "))
portf_value = float(input("Please enter the value of portfolio(Rs): "))
days = int(input("Enter the time horizon for VaR calculation: "))
confidence  = float(input("Please enter the confidence interval in decimal: "))
print("-"*100)

In [ ]:
# TIME RANGE
end_date   = dt.datetime.now()
start_date = end_date - dt.timedelta(days=365 * years)

In [ ]:
# DATA
price = dwn_adj_close_price(tickers, start_date, end_date)

In [ ]:
# LOG RETURNS
log_returns = cal_logs(price)
print("-"*10, "\tLOG RETURNS\t", "-"*10, "\n")
print(log_returns)

In [ ]:
# PORTFOLIO WEIGHTS (EQUALLY DISTRIBUTED)
weight = np.array([1/len(tickers)] * len(tickers))
print("-"*10, "\t WEIGHT DISTRIBUTION\t", "-"*10)
print(weight)

In [ ]:
# HISTORICAL RETURNS

historical_returns = get_hist_returns(log_returns, weight)

In [ ]:
# X-DAY RETURNS

xday_returns = xday_return_calc(historical_returns, days)

In [ ]:
# VaR CALCULATION
VAR_s = cal_var_s(xday_returns, confidence, portf_value)

In [ ]:
# BACKTEST
backtest_df, observations, breaches = rolling_var_backtest(
    xday_retn=xday_returns,
    p_value=portf_value,
    c=confidence
)

In [ ]:
# OUTPUTS
print("\n\n", "-"*50, " The calculated results ", "-"*50, "\n")
print(f"VaR (Historical Simulation) for portfolio of ₹{portf_value:,.2f} = ₹{VAR_s:,.2f}")
print(f"At {confidence*100:.0f}% confidence, the portfolio will not lose more than ₹{VAR_s:,.2f} in {days} day(s).")
print(f"Risk = {(VAR_s/portf_value)*100:.2f}% of total portfolio value.")
print("\n")
print("-"*20, "Breach Test", "-"*20)
print("Total Observations:", observations)
print("Breaches:", breaches)
print(f"Expected Breaches: {(observations * (1 - confidence)):.2f}")
print(f"Breach Rate: {(breaches/observations)*100:.2f}% of total observations.")

In [ ]:
#VISUALISATION

import matplotlib.pyplot as plt

xday_returns_rs = xday_returns * portf_value


plt.hist(xday_returns_rs.dropna(), bins = 100, density = True)
plt.xlabel(f"{days}-Day Portfolio Return(Rs)")
plt.ylabel("Frequency")
plt.title(f"Distribution of Portfolio {days}-Day Returns(Rs)")
plt.axvline(-VAR_s, color="#CECECE", linestyle = "dashed", linewidth = 2, label = f"VaR(historical) at {confidence:.0%} confidence level")
plt.legend()
plt.savefig('hist.png')
plt.show()